# 08 - Build Final Artifacts (Regenerasi `_fix_final/`)## TujuanMenghitung artefak final yang dipakai dashboard dan skripsi, dengan**inferensi penuh terhadap seluruh 285 dokumen** sehingga hasilnyadapat direproduksi ulang setiap kali notebook ini dijalankan.## Asal (Input)- `model/lda_model.gensim` (dari Notebook 04)- `data/intermediate/` (dari Notebook 03)## Keluaran (Output)- `topic_distribution_final.csv` - 285 dokumen + matriks theta lengkap- `topic_counts_final.csv` - jumlah & persentase per topik- `topic_by_year_final.csv` - matriks tahun x topik + proporsi- `top_words_final.csv` - kata kunci per topik- `evaluation_summary.csv` - rekap metrik model## Mengapa Inferensi Penuh?Notebook 04 melatih model pada 228 dokumen (80%) lalu memprediksi 285.Dokumen di hold-out (20%) belum pernah dilihat model, sehingga theta-nyadiffuse dan beberapa dokumen "lompat" ke topik berbeda.Notebook ini menghilangkan bias itu: seluruh 285 dokumen dinilai oleh modelsehingga setiap dokumen mendapat penugasan topik yang konsisten. Ini jugamenjadikan angka di Tabel 5 dan 13 skripsi dapat direproduksi ulang.

In [ ]:
import sys, io as _iofrom pathlib import Path_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(_ROOT))try:    sys.stdout.reconfigure(encoding='utf-8', errors='replace')except Exception:    passimport pandas as pd, numpy as npfrom src import config as CC.ensure_dirs()print(C.describe())

## Langkah 1 - Bangun Artefak Final

In [ ]:
from src import build_final as bfresult = bf.run(write=True, verbose=True)

## Langkah 2 - Verifikasi Konsistensi

In [ ]:
counts = result['counts']by_year = result['by_year']print(f'Total dokumen      : {int(counts["jumlah_dokumen"].sum())}')print(f'Total persentase    : {counts["persentase"].sum():.2f}%')print()print('Proporsi per tahun (harus ~100%):')for _, r in by_year.iterrows():    s = sum(r[f'proporsi_topik_{t}'] for t in range(1, 9))    print(f'  {int(r["Tahun"])}: {int(r["total_dokumen"]):3d} dokumen  sum={s:.2f}%')

## Langkah 3 - Metrik Model (dapat direproduksi)

In [ ]:
ev = result['evaluation']print(ev.to_string(index=False, max_colwidth=70))

## Langkah 4 - Visualisasi Distribusi Topik

In [ ]:
import matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltc = result['counts']fig, ax = plt.subplots(figsize=(8, 4.5))bars = ax.bar([f'Topik {int(t)}' for t in c['topik_dominan']],              c['jumlah_dokumen'], color='#1A56A0', alpha=0.85)for bar, val in zip(bars, c['jumlah_dokumen']):    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,            str(int(val)), ha='center', fontweight='bold')ax.set_ylabel('Jumlah Dokumen')ax.set_title(f'Distribusi Dokumen per Topik (Total {int(c["jumlah_dokumen"].sum())})')ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

## Ringkasan Fase 08 - Build Final ArtifactsSeluruh artefak di `_fix_final/` kini **dapat direproduksi ulang** denganmenjalankan notebook ini. Tidak ada lagi angka yang muncul "dari mana-mana".| File | Jumlah Baris | Status ||:---|---:|:---|| `topic_distribution_final.csv` | 285 | dapat direproduksi || `topic_counts_final.csv` | 8 | dapat direproduksi || `topic_by_year_final.csv` | 5 | dapat direproduksi || `top_words_final.csv` | 120 | dapat direproduksi || `evaluation_summary.csv` | 9 metrik | dapat direproduksi |**Catatan metodologis untuk Bab III:** distribusi final dihitung denganinferensi penuh terhadap seluruh 285 dokumen (bukan model yang di-split80/20), sehingga setiap dokumen dinilai secara optimal.

## Peta Artefak| Artefak | Dibuat oleh | Consumed by ||:---|:---|:---|| `_fix_final/topic_distribution_final.csv` | Notebook 08 | app, tests || `_fix_final/topic_counts_final.csv` | Notebook 08 | app (Beranda, Distribusi) || `_fix_final/topic_by_year_final.csv` | Notebook 08 | Notebook 06, app (Tren) || `_fix_final/top_words_final.csv` | Notebook 08 | Notebook 07 (labeling) |